## Game Logs

In [1]:
import requests
import time
import os
from bs4 import BeautifulSoup, Comment
import pandas as pd

def fetch_url(url, max_retries=5, backoff_factor=5):
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/103.0.0.0 Safari/537.36')
    }
    for i in range(max_retries):
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response
        elif response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                wait_time = int(retry_after)
            else:
                wait_time = backoff_factor * (i + 1)
            print(f"429 received. Backing off for {wait_time} seconds.")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code} received for URL: {url}")
            break
    return response

url = "https://www.basketball-reference.com/teams/LAC/2024/gamelog/"
response = fetch_url(url)
if response.status_code == 200:
    print("Page fetched successfully.")
else:
    print(f"Failed to fetch page. Status code: {response.status_code}")

def get_table(soup, table_id):
    """
    Try to locate a table by its id, looking in both the parsed HTML and any HTML comments.
    """
    table = soup.find("table", id=table_id)
    if table is None:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if f'id="{table_id}"' in comment:
                comment_soup = BeautifulSoup(comment, "html.parser")
                table = comment_soup.find("table", id=table_id)
                if table:
                    break
    return table

def parse_table(table, opp_start_idx=30):
    """
    Extract headers and data rows from the table into a DataFrame.
    Columns from the 31st position onward get an '_opp' suffix.
    """
    header_rows = table.find("thead").find_all("tr")
    header_row = header_rows[-1]
    headers = [th.get_text(strip=True) for th in header_row.find_all("th")]
    if headers and headers[0] == "":
        headers[0] = "Rk"
    if len(headers) > opp_start_idx:
        headers = headers[:opp_start_idx] + [h + "_opp" for h in headers[opp_start_idx:]]
    data = []
    for row in table.find("tbody").find_all("tr"):
        if row.get("class") and "thead" in row.get("class"):
            continue
        row_data = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if row_data:
            data.append(row_data)
    return pd.DataFrame(data, columns=headers)

def scrape_team_season(team_code, season):
    url = f"https://www.basketball-reference.com/teams/{team_code}/{season}/gamelog/"
    print(f"Scraping: {url}")
    response = fetch_url(url)
    if response.status_code != 200:
        print(f"Failed to fetch URL: {url}, status code: {response.status_code}")
        return None, None
    soup = BeautifulSoup(response.text, 'html.parser')
    
    table_reg = get_table(soup, "team_game_log_reg")
    df_reg = parse_table(table_reg) if table_reg else None
    table_post = get_table(soup, "team_game_log_post")
    df_post = parse_table(table_post) if table_post else None
    return df_reg, df_post

def main():
    # All NBA team codes on Basketball Reference
    team_codes = [
        "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", 
        "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", 
        "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", 
        "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
    ]
    # Seasons from 2022 to 2025
    seasons = ["2022", "2023", "2024", "2025"]
    
    out_dir = "scraped_gamelogs"
    os.makedirs(out_dir, exist_ok=True)
    
    for team in team_codes:
        for season in seasons:
            print(f"\nProcessing Team: {team}, Season: {season}")
            df_reg, df_post = scrape_team_season(team, season)
            
            if df_reg is not None:
                filename_reg = os.path.join(out_dir, f"gamelog_{team}_{season}_reg.csv")
                df_reg.to_csv(filename_reg, index=False)
                print(f"Saved regular season data to: {filename_reg}")
            else:
                print(f"No regular season data found for {team} in {season}.")
            
            if df_post is not None:
                filename_post = os.path.join(out_dir, f"gamelog_{team}_{season}_post.csv")
                df_post.to_csv(filename_post, index=False)
                print(f"Saved postseason data to: {filename_post}")
            else:
                print(f"No postseason data found for {team} in {season}.")
            
            # Extra pause between requests to be even more respectful of the server.
            time.sleep(2)

if __name__ == "__main__":
    main()

Page fetched successfully.

Processing Team: ATL, Season: 2022
Scraping: https://www.basketball-reference.com/teams/ATL/2022/gamelog/
Saved regular season data to: scraped_gamelogs\gamelog_ATL_2022_reg.csv
Saved postseason data to: scraped_gamelogs\gamelog_ATL_2022_post.csv

Processing Team: ATL, Season: 2023
Scraping: https://www.basketball-reference.com/teams/ATL/2023/gamelog/
Saved regular season data to: scraped_gamelogs\gamelog_ATL_2023_reg.csv
Saved postseason data to: scraped_gamelogs\gamelog_ATL_2023_post.csv

Processing Team: ATL, Season: 2024
Scraping: https://www.basketball-reference.com/teams/ATL/2024/gamelog/
Saved regular season data to: scraped_gamelogs\gamelog_ATL_2024_reg.csv
No postseason data found for ATL in 2024.

Processing Team: ATL, Season: 2025
Scraping: https://www.basketball-reference.com/teams/ATL/2025/gamelog/
Saved regular season data to: scraped_gamelogs\gamelog_ATL_2025_reg.csv
No postseason data found for ATL in 2025.

Processing Team: BOS, Season: 2022

## Advanced Game Logs

In [8]:
import requests
import time
import os
from bs4 import BeautifulSoup, Comment
import pandas as pd

def fetch_url(url, max_retries=5, backoff_factor=5):
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/103.0.0.0 Safari/537.36')
    }
    for i in range(max_retries):
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response
        elif response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            wait_time = int(retry_after) if retry_after else backoff_factor * (i + 1)
            print(f"429 received. Backing off for {wait_time} seconds.")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code} received for URL: {url}")
            break
    return response

def get_table(soup, table_id):
    table = soup.find("table", id=table_id)
    if table is None:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if f'id="{table_id}"' in comment:
                comment_soup = BeautifulSoup(comment, "html.parser")
                table = comment_soup.find("table", id=table_id)
                if table:
                    break
    return table

def parse_table(table):
    header_rows = table.find("thead").find_all("tr")
    header_row = header_rows[-1]
    headers = [th.get_text(strip=True) for th in header_row.find_all("th")]
    if headers and headers[0] == "":
        headers[0] = "Rk"

    # Modify only last 4 columns to "opp"
    if len(headers) > 4:
        headers[-4:] = [h.replace("_opp", "opp") for h in headers[-4:]]

    data = [[td.get_text(strip=True) for td in row.find_all(["th", "td"])] 
            for row in table.find("tbody").find_all("tr") if "thead" not in row.get("class", [])]
    
    return pd.DataFrame(data, columns=headers)

def scrape_team_season(team_code, season):
    url = f"https://www.basketball-reference.com/teams/{team_code}/{season}/gamelog-advanced/"
    response = fetch_url(url)
    if response.status_code != 200:
        print(f"Failed to fetch URL: {url}, status code: {response.status_code}")
        return None, None
    soup = BeautifulSoup(response.text, 'html.parser')

    table_reg = get_table(soup, "team_game_log_adv_reg")
    df_reg = parse_table(table_reg) if table_reg else None
    table_post = get_table(soup, "team_game_log_adv_post")
    df_post = parse_table(table_post) if table_post else None
    return df_reg, df_post

def main():
    team_codes = [
        "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", 
        "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", 
        "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", 
        "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
    ]
    seasons = ["2022", "2023", "2024", "2025"]

    out_dir = "scraped_gamelogs_advanced"
    os.makedirs(out_dir, exist_ok=True)

    for team in team_codes:
        for season in seasons:
            print(f"\nProcessing Team: {team}, Season: {season}")
            df_reg, df_post = scrape_team_season(team, season)

            if df_reg is not None:
                filename_reg = os.path.join(out_dir, f"gamelog_advanced_{team}_{season}_reg.csv")
                df_reg.to_csv(filename_reg, index=False)
                print(f"Saved regular season data to: {filename_reg}")
            else:
                print(f"No regular season data found for {team} in {season}.")

            if df_post is not None:
                filename_post = os.path.join(out_dir, f"gamelog_advanced_{team}_{season}_post.csv")
                df_post.to_csv(filename_post, index=False)
                print(f"Saved postseason data to: {filename_post}")
            else:
                print(f"No postseason data found for {team} in {season}.")

            time.sleep(2)  # Respectful pause

if __name__ == "__main__":
    main()


Processing Team: ATL, Season: 2022
Saved regular season data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2022_reg.csv
Saved postseason data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2022_post.csv

Processing Team: ATL, Season: 2023
Saved regular season data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2023_reg.csv
Saved postseason data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2023_post.csv

Processing Team: ATL, Season: 2024
Saved regular season data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2024_reg.csv
No postseason data found for ATL in 2024.

Processing Team: ATL, Season: 2025
Saved regular season data to: scraped_gamelogs_advanced\gamelog_advanced_ATL_2025_reg.csv
No postseason data found for ATL in 2025.

Processing Team: BOS, Season: 2022
Saved regular season data to: scraped_gamelogs_advanced\gamelog_advanced_BOS_2022_reg.csv
Saved postseason data to: scraped_gamelogs_advanced\gamelog_advanced_BOS_2022_post.csv

Processing Team: BOS, Se

## Player Stats

In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

def fetch_url(url, max_retries=5, backoff_factor=5):
    headers = {
        'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                       'AppleWebKit/537.36 (KHTML, like Gecko) '
                       'Chrome/103.0.0.0 Safari/537.36')
    }
    for i in range(max_retries):
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response
        elif response.status_code == 429:
            retry_after = response.headers.get("Retry-After")
            if retry_after:
                wait_time = int(retry_after)
            else:
                wait_time = backoff_factor * (i + 1)
            print(f"429 received. Backing off for {wait_time} seconds.")
            time.sleep(wait_time)
        else:
            print(f"Error {response.status_code} received for URL: {url}")
            break
    return response

# Define all NBA team codes as used on Basketball Reference
team_codes = [
    "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", 
    "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", 
    "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
]

# Define seasons to scrape
seasons = ["2022", "2023", "2024", "2025"]

# Define output directory
out_dir = "scraped_team_stats"
os.makedirs(out_dir, exist_ok=True)

def get_table(soup, table_id):
    """
    Find a table using its ID in a BeautifulSoup object.
    """
    table = soup.find("table", id=table_id)
    return table

def parse_table(table):
    """
    Extract the header and data rows from a table and return a DataFrame.
    """
    if not table:
        return None

    headers = [th.get_text(strip=True) for th in table.find("thead").find_all("th")]
    
    data = []
    for row in table.find("tbody").find_all("tr"):
        row_data = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if row_data:
            data.append(row_data)

    return pd.DataFrame(data, columns=headers)

def scrape_team_stats(team_code, season):
    """
    Scrape per-game and postseason per-game statistics for a given team and season.
    """
    url = f"https://www.basketball-reference.com/teams/{team_code}/{season}.html"
    print(f"Scraping: {url}")
    
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {url}, status code: {response.status_code}")
        return None, None

    soup = BeautifulSoup(response.text, "html.parser")

    # Regular season per-game stats
    table_reg = get_table(soup, "per_game_stats")
    df_reg = parse_table(table_reg) if table_reg else None

    # Postseason per-game stats
    table_post = get_table(soup, "per_game_stats_post")
    df_post = parse_table(table_post) if table_post else None

    return df_reg, df_post

# Loop through all teams and seasons
for team in team_codes:
    for season in seasons:
        print(f"\nProcessing {team} - {season}")

        df_reg, df_post = scrape_team_stats(team, season)

        # Create a directory for the season
        season_dir = os.path.join(out_dir, season)
        os.makedirs(season_dir, exist_ok=True)

        # Save regular season stats
        if df_reg is not None:
            reg_filename = os.path.join(season_dir, f"{team}_{season}_per_game.csv")
            df_reg.to_csv(reg_filename, index=False)
            print(f"Saved: {reg_filename}")
        else:
            print(f"No per-game stats found for {team} in {season}.")

        # Save postseason stats
        if df_post is not None:
            post_filename = os.path.join(season_dir, f"{team}_{season}_per_game_post.csv")
            df_post.to_csv(post_filename, index=False)
            print(f"Saved: {post_filename}")
        else:
            print(f"No postseason per-game stats found for {team} in {season}.")

        # Pause to avoid excessive requests
        time.sleep(1)

print("Scraping complete!")


Processing ATL - 2022
Scraping: https://www.basketball-reference.com/teams/ATL/2022.html
Saved: scraped_team_stats\2022\ATL_2022_per_game.csv
Saved: scraped_team_stats\2022\ATL_2022_per_game_post.csv

Processing ATL - 2023
Scraping: https://www.basketball-reference.com/teams/ATL/2023.html
Saved: scraped_team_stats\2023\ATL_2023_per_game.csv
Saved: scraped_team_stats\2023\ATL_2023_per_game_post.csv

Processing ATL - 2024
Scraping: https://www.basketball-reference.com/teams/ATL/2024.html
Saved: scraped_team_stats\2024\ATL_2024_per_game.csv
No postseason per-game stats found for ATL in 2024.

Processing ATL - 2025
Scraping: https://www.basketball-reference.com/teams/ATL/2025.html
No per-game stats found for ATL in 2025.
No postseason per-game stats found for ATL in 2025.

Processing BOS - 2022
Scraping: https://www.basketball-reference.com/teams/BOS/2022.html
Saved: scraped_team_stats\2022\BOS_2022_per_game.csv
Saved: scraped_team_stats\2022\BOS_2022_per_game_post.csv

Processing BOS - 2

KeyboardInterrupt: 

## Advanced Player Stats

In [3]:
import requests
from bs4 import BeautifulSoup, Comment
import pandas as pd
import os
import time

# Define all NBA team codes as used on Basketball Reference
team_codes = [
    "ATL", "BOS", "BRK", "CHO", "CHI", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", 
    "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", 
    "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
]

# Define seasons to scrape
seasons = ["2022", "2023", "2024", "2025"]

# Define output directory
out_dir = "scraped_team_stats_advanced"
os.makedirs(out_dir, exist_ok=True)

def get_table(soup, table_id):
    """
    Attempts to locate a table by its id.
    If not found directly, it checks inside HTML comments.
    """
    # First, try to find the table directly
    table = soup.find("table", id=table_id)
    if table:
        return table

    # If not found, search inside all the HTML comments
    comments = soup.find_all(string=lambda text: isinstance(text, Comment))
    for comment in comments:
        if f'id="{table_id}"' in comment:
            comment_soup = BeautifulSoup(comment, "html.parser")
            table = comment_soup.find("table", id=table_id)
            if table:
                return table
    return None

def parse_table(table):
    """
    Extract headers and rows from the provided table and return a DataFrame.
    """
    if table is None:
        return None
    
    # Get header row (assumes the first row in <thead> holds headers)
    header_row = table.find("thead").find("tr")
    headers = [th.get_text(strip=True) for th in header_row.find_all("th")]

    # Extract body rows
    data = []
    for row in table.find("tbody").find_all("tr"):
        # Sometimes Basketball Reference includes extra header rows inside tbody
        if row.get("class") and "thead" in row.get("class"):
            continue
        row_data = [td.get_text(strip=True) for td in row.find_all(["th", "td"])]
        if row_data:
            data.append(row_data)
    
    return pd.DataFrame(data, columns=headers)

def scrape_team_stats_advanced(team_code, season):
    """
    Scrapes the advanced per-game stats and postseason advanced stats for a given team and season.
    It looks for tables with ids "advanced" and "advanced_post".
    """
    url = f"https://www.basketball-reference.com/teams/{team_code}/{season}.html"
    print(f"Scraping: {url}")
    
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Failed to fetch {url}, status code: {response.status_code}")
        return None, None

    soup = BeautifulSoup(response.text, "html.parser")

    # Regular season advanced stats table (found under the anchor "#advanced")
    table_reg = get_table(soup, "advanced")
    df_reg = parse_table(table_reg) if table_reg else None

    # Postseason advanced stats table (found under the anchor "#advanced_post")
    table_post = get_table(soup, "advanced_post")
    df_post = parse_table(table_post) if table_post else None

    return df_reg, df_post

# Loop through all teams and seasons
for team in team_codes:
    for season in seasons:
        print(f"\nProcessing {team} - {season}")

        df_reg, df_post = scrape_team_stats_advanced(team, season)

        # Create a directory for the season
        season_dir = os.path.join(out_dir, season)
        os.makedirs(season_dir, exist_ok=True)

        # Save the regular season advanced stats if available
        if df_reg is not None:
            reg_filename = os.path.join(season_dir, f"{team}_{season}_advanced.csv")
            df_reg.to_csv(reg_filename, index=False)
            print(f"Saved regular season advanced stats: {reg_filename}")
        else:
            print(f"No regular season advanced stats found for {team} in {season}.")

        # Save the postseason advanced stats if available
        if df_post is not None:
            post_filename = os.path.join(season_dir, f"{team}_{season}_advanced_post.csv")
            df_post.to_csv(post_filename, index=False)
            print(f"Saved postseason advanced stats: {post_filename}")
        else:
            print(f"No postseason advanced stats found for {team} in {season}.")

        # Pause to avoid making too many rapid requests
        time.sleep(1)

print("Scraping complete!")


Processing ATL - 2022
Scraping: https://www.basketball-reference.com/teams/ATL/2022.html
Saved regular season advanced stats: scraped_team_stats_advanced\2022\ATL_2022_advanced.csv
Saved postseason advanced stats: scraped_team_stats_advanced\2022\ATL_2022_advanced_post.csv

Processing ATL - 2023
Scraping: https://www.basketball-reference.com/teams/ATL/2023.html
Saved regular season advanced stats: scraped_team_stats_advanced\2023\ATL_2023_advanced.csv
Saved postseason advanced stats: scraped_team_stats_advanced\2023\ATL_2023_advanced_post.csv

Processing ATL - 2024
Scraping: https://www.basketball-reference.com/teams/ATL/2024.html
Saved regular season advanced stats: scraped_team_stats_advanced\2024\ATL_2024_advanced.csv
No postseason advanced stats found for ATL in 2024.

Processing ATL - 2025
Scraping: https://www.basketball-reference.com/teams/ATL/2025.html
Saved regular season advanced stats: scraped_team_stats_advanced\2025\ATL_2025_advanced.csv
No postseason advanced stats found

KeyboardInterrupt: 

## Team Logos

Team logo for Charlotte Hornets was set to 'CHA' instead of 'CHO' so grabbed seperately then renamed the file.

In [3]:
import os
import time
import requests
from PIL import Image
from io import BytesIO

# Create directory to store logos
os.makedirs("team_logos", exist_ok=True)

# NBA team codes from Basketball Reference
team_codes = [
    "ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW",
    "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK",
    "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"
]

# Logo URL template
base_url = "https://cdn.ssref.net/req/202506181/tlogo/bbr/{}.png"

for code in team_codes:
    url = base_url.format(code)
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            img.save(f"team_logos/{code}.png")
            print(f"Downloaded: {code}")
        else:
            print(f"Failed for {code}: status {response.status_code}")
    except Exception as e:
        print(f"Error fetching {code}: {e}")
    
    time.sleep(1.5)  # Pause between requests (1.5 seconds recommended)

Downloaded: ATL
Downloaded: BOS
Downloaded: BRK
Downloaded: CHI
Failed for CHO: status 404
Downloaded: CLE
Downloaded: DAL
Downloaded: DEN
Downloaded: DET
Downloaded: GSW
Downloaded: HOU
Downloaded: IND
Downloaded: LAC
Downloaded: LAL
Downloaded: MEM
Downloaded: MIA
Downloaded: MIL
Downloaded: MIN
Downloaded: NOP
Downloaded: NYK
Downloaded: OKC
Downloaded: ORL
Downloaded: PHI
Downloaded: PHO
Downloaded: POR
Downloaded: SAC
Downloaded: SAS
Downloaded: TOR
Downloaded: UTA
Downloaded: WAS


In [4]:
import os
import time
import requests
from PIL import Image
from io import BytesIO

# Create directory to store logos
os.makedirs("team_logos", exist_ok=True)

# NBA team codes from Basketball Reference
team_codes = [
    "CHA"
]

# Logo URL template
base_url = "https://cdn.ssref.net/req/202506181/tlogo/bbr/{}.png"

for code in team_codes:
    url = base_url.format(code)
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            img = Image.open(BytesIO(response.content))
            img.save(f"team_logos/{code}.png")
            print(f"Downloaded: {code}")
        else:
            print(f"Failed for {code}: status {response.status_code}")
    except Exception as e:
        print(f"Error fetching {code}: {e}")
    
    time.sleep(1.5)  # Pause between requests (1.5 seconds recommended)

Downloaded: CHA
